[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# Shapely Geometries &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import LineString, MultiPoint, Point, Polygon
import matplotlib.pyplot as plt

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


norway = countries.loc[countries["ADMIN"] == "Norway", "geometry"].iloc[0]
print("shapely", shapely.__version__, "| GEOS", shapely.geos_version_string, "| Norway is a",
      norway.geom_type, "of", len(norway.geoms), "parts")

shapely 2.2.0 | GEOS 3.14.1 | Norway is a MultiPolygon of 31 parts


The worked examples' grounds, rebuilt here so that each answer runs on its own. The grounds are made up.

In [2]:
grounds = Polygon(
    [(0, 0), (120, 0), (120, 80), (0, 80)],
    holes=[[(80, 20), (100, 20), (100, 40), (80, 40)]],
)
norway_m = countries.to_crs(25833).loc[countries["ADMIN"] == "Norway", "geometry"].iloc[0]

**1.** Make a point for Tromso from `weather_stations`, print it as WKT, and print whether it is within `norway`.

In [3]:
row = weather_stations[weather_stations["name"] == "Tromso"].iloc[0]
tromso = Point(row["longitude"], row["latitude"])
print(tromso)
print("within Norway's outline:", tromso.within(norway))

POINT (18.96 69.65)
within Norway's outline: False


Tromso is not within the outline, as the **Why GeoPandas** notebook found. The town stands on the coast, and the 1:50m outline smooths the coastline and leaves it just outside.

**2.** Build a `LineString` from Bergen to Oslo to Tromso, in that order, from `weather_stations`, and print how many coordinates it has and its bounds.

In [4]:
order = weather_stations.set_index("name").loc[["Bergen", "Oslo", "Tromso"]]
route = LineString(list(zip(order["longitude"], order["latitude"])))
print(len(route.coords), "coordinates")
print(route.bounds)

3 coordinates
(5.32, 59.91, 18.96, 69.65)


`set_index` and `loc` put the stations in the order asked for, and longitude goes first in each pair. The bounds are in degrees, the units of the coordinates.

**3.** Make a rectangle 30 m wide and 50 m tall with `shapely.box`, and print its area and its perimeter.

In [5]:
plot = shapely.box(0, 0, 30, 50)
print("area:", plot.area, "m² | perimeter:", plot.length, "m")

area: 1500.0 m² | perimeter: 160.0 m


`box` takes the bounds, `(min_x, min_y, max_x, max_y)`, so a box from (0, 0) to (30, 50) is 30 m wide and 50 m tall.

**4.** Test which of the points (10, 70), (90, 30) and (130, 40) lie within `grounds`, and print a list of the answers.

In [6]:
points = [Point(10, 70), Point(90, 30), Point(130, 40)]
print([point.within(grounds) for point in points])

[True, False, False]


The first point is inside the fence, the second is in the pond, which is not part of the grounds, and the third is east of the fence.

**5.** Count the parts of Norway's outline that measure more than 1,000 km², using `norway_m`.

In [7]:
large = [part for part in norway_m.geoms if part.area / 1e6 > 1000]
print(len(large), "of", len(norway_m.geoms), "parts")

8 of 31 parts


`norway_m` is in meters, so an area divided by a million is in square kilometers.

---

&#8592; **Back to:** [Shapely Geometries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/02-shapely-geometries.ipynb)  &nbsp;&middot;&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)
